In [33]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from units.df_func import calculate_wqi_for_row
import folium

In [26]:
df = pd.read_excel('data/test.xlsx')

In [29]:
df['geometry'] = df.apply(lambda row: Point(row['X'], row['Y']), axis=1)
gdf = gpd.GeoDataFrame(df, geometry='geometry')
gdf.crs = "EPSG:4326" 

In [30]:
gdf['wqi'] = df.apply(lambda row: calculate_wqi_for_row(row, weighted=False), axis=1)

In [ ]:
color = ''

In [40]:
def get_color(wqi):
    if wqi <= 50:
        return 'red'
    elif wqi <= 75:
        return 'orange'
    else:
        return 'green'

In [46]:
# Lấy tọa độ trung tâm để đặt vị trí ban đầu của bản đồ
center_lat = gdf.geometry.centroid.y.mean()
center_lon = gdf.geometry.centroid.x.mean()

# Tạo bản đồ folium
m = folium.Map(location=[center_lat, center_lon], zoom_start=10)

# Hàm để chọn màu dựa trên giá trị WQI
def get_color(wqi):
    if wqi <= 50:
        return 'red'
    elif wqi <= 75:
        return 'orange'
    else:
        return 'green'

# Hàm để chọn bán kính dựa trên giá trị WQI (giá trị thấp có bán kính lớn hơn)
def get_radius(wqi):
    if wqi <= 50:
        return 20  # Bán kính lớn cho WQI thấp
    elif wqi <= 75:
        return 15  # Bán kính trung bình
    else:
        return 10  # Bán kính nhỏ cho WQI cao

# Thêm CircleMarker vào bản đồ với màu sắc và bán kính tùy chỉnh
for _, row in gdf.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=get_radius(row['wqi']),  # Bán kính tùy chỉnh dựa trên WQI
        color=get_color(row['wqi']),
        fill=True,
        fill_color=get_color(row['wqi']),
        fill_opacity=0.7,
        popup=f"Tên trạm: {row['Tên trạm']}<br>WQI: {row['wqi']}"
    ).add_to(m)

# Lưu bản đồ thành file HTML
m.save("map_wqi_radius.html")

C:\Users\thuy\AppData\Local\Temp\ipykernel_31248\599375143.py:2: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center_lat = gdf.geometry.centroid.y.mean()
C:\Users\thuy\AppData\Local\Temp\ipykernel_31248\599375143.py:3: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center_lon = gdf.geometry.centroid.x.mean()


In [47]:
display(m)